# 03-3 · The lambda dial — does it move the pick?

| | |
|---|---|
| **What it does** | sweeps lambda from 0 to 100 on the real catalog, then on a constructed catalog designed to conflict |
| **Proves / disproves** | Figure 1's claim only holds where mean and variance actually trade off — measured here, not assumed |
| **Uses** | `nbio.fairness_score`, `nbio.aggregate` (Vincent's code) |
| **Eval surface** | the crossing point where `fair(lambda)` stops agreeing with `average` |

No API key needed.


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)


In [2]:
import nbio, json
nbio.bootstrap()


repo     : /project/painting/local-projects-to-run/research/ai-in-food/fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/local-projects-to-run/research/ai-in-food/fair-mediator')

## Rebuild F, S on the real catalog (same 4 members as 03-1/03-2)


In [3]:
# Step 1: rebuild
personas = json.load(open(nbio.DATA / "personas.json"))
group_names = ["Vihaan", "Vincent", "Vasu", "Alice"]
members = []
for name in group_names:
    p = next(x for x in personas if x["display_name"] == name)
    members.append({"member_id": name.lower(), "display_name": name,
                     "diet": p["diet_type"], "allergies": p["allergies"], "taste": p["taste_vector"]})

import csv
flavor_map = {}
with open(nbio.DATA / "ingredient-flavor.csv") as f:
    for row in csv.DictReader(f):
        n = row.get("ingredient", "").strip().lower()
        if n: flavor_map[n] = {d: float(row.get(d, 0) or 0) for d in nbio.TASTE_DIMS}

def iter_dish_names(node):
    if isinstance(node, dict):
        if ("item" in node or "name" in node) and "price" in node:
            yield (node.get("item") or node.get("name") or "").strip(); return
        for v in node.values(): yield from iter_dish_names(v)
    elif isinstance(node, list):
        for v in node: yield from iter_dish_names(v)

def infer_taste(d):
    m = [flavor_map[i] for i in flavor_map if i in d.lower()]
    return [sum(x[k] for x in m)/len(m) for k in nbio.TASTE_DIMS] if m else [0.0]*6

catalog = []
for f in sorted((nbio.DATA / "Menu").glob("*.json")):
    names = [n for n in iter_dish_names(json.loads(f.read_text())) if n]
    vecs = [infer_taste(n) for n in names]
    nz = [v for v in vecs if sum(abs(x) for x in v) > 0]
    avg = [sum(v[i] for v in nz)/len(nz) for i in range(6)] if nz else [0.0]*6
    catalog.append({"option_id": f.stem, "name": f.stem, "menu": [{"name": n} for n in names], "taste": avg})

F = nbio.build_feasible_set(catalog, members)
candidate_vectors = {o["option_id"]: o["taste"] for o in F}
member_vectors = [m["taste"] for m in members]
labels = {o["option_id"]: o["name"] for o in F}
print(f"F size: {len(F)}")


F size: 5


## Sweep lambda on the real catalog


In [4]:
# Step 2: does raising lambda ever change the pick?
seen = {}
for lam in [0, 0.5, 1, 2, 5, 10, 25, 50, 100]:
    ranked = nbio.rank_candidates(candidate_vectors, member_vectors, method="fair", lam=lam)
    winner_id, score = ranked[0]
    seen.setdefault(labels[winner_id], []).append(lam)
    print(f"   lambda={lam:<5} -> {labels[winner_id]:<32} score={score:+.4f}")

print()
print("distinct picks across lambda:", seen)


   lambda=0     -> Casa.cafe                        score=+0.7609
   lambda=0.5   -> Casa.cafe                        score=+0.7571
   lambda=1     -> Casa.cafe                        score=+0.7533
   lambda=2     -> Casa.cafe                        score=+0.7456
   lambda=5     -> Casa.cafe                        score=+0.7225
   lambda=10    -> Casa.cafe                        score=+0.6840
   lambda=25    -> Delicia Multi Cuisine Restaurant score=+0.5709
   lambda=50    -> Delicia Multi Cuisine Restaurant score=+0.3832
   lambda=100   -> Delicia Multi Cuisine Restaurant score=+0.0077

distinct picks across lambda: {'Casa.cafe': [0, 0.5, 1, 2, 5, 10], 'Delicia Multi Cuisine Restaurant': [25, 50, 100]}


> **Measured on this 4-member group: the pick DOES move, but only at a large
> lambda.** `fair(lambda)` agrees with `average` (picks Casa.cafe) up to
> lambda=10, then crosses over to Delicia Multi Cuisine at lambda=25 and stays
> there through 100. The dial is not inert here — it just needs an unusually
> large fairness penalty to overturn the mean-maximiser, which itself is a
> measurement: the mean and variance objectives are only weakly in tension on
> this catalog. A catalog with more genuine spread (PRD-4) should show the
> crossover at a smaller, more interpretable lambda.


## Does the mechanism itself work? A constructed conflict

Three diners in the shape of the paper's own worked example: two want heat, one
wants umami, and — this is the deliberate design choice — **no balanced option
exists**. With cosine similarity a balanced option would dominate on mean AND
minimum simultaneously, so a genuine conflict requires its absence.


In [5]:
# Step 3: a catalog built to make Average and Least-Misery disagree
toy_members = [
    {"member_id": "ann",  "display_name": "Ann",  "taste": [0, 0, 0, 0, 0.0, 1.0]},
    {"member_id": "ben",  "display_name": "Ben",  "taste": [0, 0, 0, 0, 0.2, 0.98]},
    {"member_id": "cara", "display_name": "Cara", "taste": [0, 0, 0, 0, 1.0, 0.0]},
]
toy_catalog = {
    "chilli_house": [0, 0, 0, 0, 0.0, 1.0],
    "umami_den":    [0, 0, 0, 0, 0.95, 0.31],
    "sweet_stop":   [1.0, 0, 0, 0, 0.0, 0.0],
}
toy_member_vecs = [m["taste"] for m in toy_members]

seen = {}
for lam in [0, 0.25, 0.5, 0.75, 1, 2, 4]:
    ranked = nbio.rank_candidates(toy_catalog, toy_member_vecs, method="fair", lam=lam)
    winner, score = ranked[0]
    seen.setdefault(winner, []).append(lam)
    print(f"   lambda={lam:<5} -> {winner:<14} score={score:+.4f}")
print()
print("lambda -> winner:", seen)


   lambda=0     -> chilli_house   score=+0.6599
   lambda=0.25  -> chilli_house   score=+0.6055
   lambda=0.5   -> chilli_house   score=+0.5510
   lambda=0.75  -> umami_den      score=+0.5306
   lambda=1     -> umami_den      score=+0.5125
   lambda=2     -> umami_den      score=+0.4400
   lambda=4     -> umami_den      score=+0.2950

lambda -> winner: {'chilli_house': [0, 0.25, 0.5], 'umami_den': [0.75, 1, 2, 4]}


In [6]:
# Step 4: assert the machinery genuinely responds to a real conflict
avg_pick = nbio.rank_candidates(toy_catalog, toy_member_vecs, method="average")[0][0]
lm_pick  = nbio.rank_candidates(toy_catalog, toy_member_vecs, method="least_misery")[0][0]
lo_pick  = nbio.rank_candidates(toy_catalog, toy_member_vecs, method="fair", lam=0.0)[0][0]
hi_pick  = nbio.rank_candidates(toy_catalog, toy_member_vecs, method="fair", lam=4.0)[0][0]

assert avg_pick != lm_pick, "constructed case failed to create a selector conflict"
assert lo_pick != hi_pick, "constructed case failed to make lambda move the pick"
print(f"PASS -- average picks {avg_pick!r}; least_misery picks {lm_pick!r}")
print(f"PASS -- fair(0) picks {lo_pick!r}; fair(4) picks {hi_pick!r}")


PASS -- average picks 'chilli_house'; least_misery picks 'umami_den'
PASS -- fair(0) picks 'chilli_house'; fair(4) picks 'umami_den'


### Reading it

Two findings, and they are not in tension. **(1)** On the real catalog, `lambda`
does move the pick, but only once it crosses roughly 10-25 — a much larger
penalty than a reader would expect to matter, because five real restaurants
here are similar enough that the mean and variance objectives only weakly
disagree. **(2)** On a catalog constructed to contain a genuine trade-off,
`lambda` moves the pick at a small, interpretable value (between 0.5 and
0.75). Both assertions in Step 4 pass. **Consequence for Figure 1**: the crossover
point itself is a number worth reporting — it measures how much fairness
pressure this catalog needs before it matters — but a curve drawn only from this
5-restaurant catalog will look flatter than the true shape of the trade-off.
PRD-4 (menu expansion) and PRD-5 (synthetic groups with real conflicts) are what
let Figure 1 show the crossover at a realistic scale.
